# PHA 快速上手

用几行代码体验个人健康 Agent（PHA）。

## PHA 是什么？

PHA 结合三个方面分析健康数据并给出个性化见解。

- **数据分析**：识别趋势和规律，与参考人群比较。
- **医学解读**：解释各项指标的含义。
- **健康指导**：给出可执行的建议。

## 环境要求

```bash
pip install google-genai pandas openai anthropic
# 可选：安装 OneTwo，启用领域专家 Agent 的完整功能
pip install git+https://github.com/google-deepmind/onetwo
```

## 快速配置

In [ ]:
import os
import sys

# --- Configuration ---
# 在此设定您的 API 密钥和提供者
API_KEY = "API_KEY"
# 提供方选项："基因","openai","人类"
PROVIDER = "PROVIDER"
# 可选：Tavily API 键 领域专家 Agent (搜索).
TAVILY_API_KEY = "TAVILY_API_KEY"

In [ ]:
import pandas as pd
import glob
from pha.agents import (
    DataScienceAgent,
    DomainExpertAgent,
    HealthCoachAgent,
    MultiAgentOrchestrator,
    is_react_available,
)

# 装入您的健康数据(在此使用样本数据)
summary_df = pd.read_csv('../data/sample/summary.csv')
activities_df = pd.read_csv('../data/sample/activities.csv')
profile_df = pd.read_csv('../data/sample/profile.csv')
population_df = pd.read_csv('../data/sample/population_percentiles.csv')

print(f"Loaded {len(summary_df)} days of health data")
print(f"ReAct available: {is_react_available()}")


In [ ]:
# 初始化 数据科学 Agent。
ds_agent = DataScienceAgent()
ds_agent.configure(api_key=API_KEY, provider=PROVIDER)  # 明确的配置
ds_agent.load_dataframes({
    'summary': summary_df,
    'activities': activities_df,
    'profile': profile_df,
    'population': population_df,
})
print("✓ Data Science Agent ready")

# 初始化 领域专家 Agent。
de_agent = None
if is_react_available():
    de_agent = DomainExpertAgent(
        search_backend='tavily',
        tavily_api_key=TAVILY_API_KEY,
    )
    exemplar_files = glob.glob('../few_shots/*.ipynb')
    de_agent.get_agent(
        api_key=API_KEY,
        provider=PROVIDER,
        exemplar_files=exemplar_files,
    )
    print(f"✓ Domain Expert Agent ready ({len(exemplar_files)} exemplars)")
else:
    print("⚠ Domain Expert Agent unavailable (install onetwo for full support)")

# 启动卫生教练
coach = HealthCoachAgent(simple_mode=True)
coach.configure(api_key=API_KEY, provider=PROVIDER)  # 明确的配置
print("✓ Health Coach ready")

# 创建所有代理的 编排器
pha = MultiAgentOrchestrator()
pha.configure(api_key=API_KEY, provider=PROVIDER)  # 明确的配置
pha.set_agents(
    data_science_agent=ds_agent,
    domain_expert_agent=de_agent,
    health_coach_agent=coach,
)

print("\n✓ PHA ready!")


## 提出健康问题

In [ ]:
# 问一个健康问题
response = pha.respond("How has my sleep been lately?")
print(response)

In [ ]:
# 询问后续问题
response = pha.respond("What about my activity levels?")
print(response)

In [ ]:
# 获得个性化建议
response = pha.respond("What should I focus on to improve my health?")
print(response)

## 单独调用各个 Agent

也可以独立使用每个 Agent。

In [ ]:
# 数据科学 Agent - 用于数据分析
analysis = ds_agent.query("What's my average daily step count?")
print("Data Science Agent:")
print(analysis)

In [ ]:
# 健康教练 -- -- 谈话指导
coach.reset_conversation()
response = coach.respond("I want to improve my energy levels")
print("Health Coach:")
print(response)

In [ ]:
# 领域专家 -- -- 健康解释和背景
if de_agent:
    response = de_agent.call_agent("What does an HbA1c of 5.8% mean for my health?")
    print("Domain Expert Agent:")
    print(response)
else:
    print("Domain Expert not available (requires onetwo)")


## 后续阅读

- `02_data_science_agent.ipynb` 展示详细的数据分析。
- `03_domain_expert_agent.ipynb` 展示如何解读健康指标。
- `04_health_coach_agent.ipynb` 展示对话式健康指导。
- `05_full_pipeline.ipynb` 展示完整的编排流程。

## 加载自己的数据

```python
# 读取自己的健康数据
my_summary = pd.read_csv('path/to/your/summary.csv')
my_activities = pd.read_csv('path/to/your/activities.csv')

ds_agent.load_dataframes({
    'summary': my_summary,
    'activities': my_activities,
})
```

所需字段包括以下几项。

- **summary**：`datetime`、`steps`、`sleep_minutes`、`resting_heart_rate`、`heart_rate_variability` 等。
- **activities**：`start_time`、`activity_name`、`duration`、`calories`、`steps` 等。